In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Early-Sepsis-Detection"

Mounted at /content/drive
/content/drive/MyDrive/Early-Sepsis-Detection


# 09 — Preprocessing Pipeline
### Early Sepsis Detection — Phase 11

Builds a scikit-learn preprocessing pipeline (median imputation + scaling
for numeric features, most-frequent imputation + one-hot encoding for
categorical features), fit ONLY on the training split from
`08_train_val_test_split.ipynb`.

**Leakage guard:** `onset_hour`, `usable_hours`, and `n_hours_used` are
metadata from Phase 7/8's window construction and leak the label almost
perfectly if used as features (see `src/preprocessing.py` docstring for
why). They are explicitly excluded here — verified in Section 2 below.


In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
from src import preprocessing as pp

pd.set_option("display.max_columns", 60)


In [3]:
patient_df = pd.read_parquet(config.PROCESSED_PATIENT_LEVEL_PARQUET)
split_lookup = pd.read_parquet(config.PROCESSED_DIR / "patient_split_assignment.parquet")

train_ids = split_lookup.loc[split_lookup["split"] == "train", config.PATIENT_ID_COL].to_numpy()
val_ids = split_lookup.loc[split_lookup["split"] == "val", config.PATIENT_ID_COL].to_numpy()
test_ids = split_lookup.loc[split_lookup["split"] == "test", config.PATIENT_ID_COL].to_numpy()

print(f"Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")
print("Loaded patient-level dataset:", patient_df.shape)


Train: 27560 | Val: 5905 | Test: 5906
Loaded patient-level dataset: (39371, 467)


## 1. Identify feature columns (leakage-risk columns excluded)

In [4]:
feature_cols = pp.get_feature_columns(patient_df)
numeric_cols, categorical_cols = pp.get_column_groups(feature_cols)

print(f"Total feature columns: {len(feature_cols)} ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")
print()
print("Excluded (non-feature) columns:", pp.NON_FEATURE_COLS)


Total feature columns: 461 (458 numeric, 3 categorical)

Excluded (non-feature) columns: ['patient_id', 'source_set', 'label', 'onset_hour', 'usable_hours', 'n_hours_used']


## 2. MANDATORY leakage verification

Confirm the excluded columns are genuinely absent from the feature set,
and — as a double-check — confirm they WOULD have been almost perfectly
predictive of the label if included (proving why the exclusion matters).


In [5]:
for leak_col in ["onset_hour", "usable_hours", "n_hours_used"]:
    assert leak_col not in feature_cols, f"LEAKAGE: {leak_col} found in feature_cols!"
print("PASSED: onset_hour, usable_hours, n_hours_used are all absent from feature_cols.")

print()
print("Demonstration of WHY these are excluded (correlation with label if they were used):")
demo = patient_df[["label", "onset_hour", "usable_hours"]].copy()
demo["onset_hour_is_null"] = demo["onset_hour"].isna().astype(int)
print("Correlation(label, onset_hour_is_null):", demo["label"].corr(demo["onset_hour_is_null"]))
print("Correlation(label, usable_hours):", demo["label"].corr(demo["usable_hours"]))


PASSED: onset_hour, usable_hours, n_hours_used are all absent from feature_cols.

Demonstration of WHY these are excluded (correlation with label if they were used):
Correlation(label, onset_hour_is_null): -1.0
Correlation(label, usable_hours): -0.29653836177686066


## 3. Fit the preprocessing pipeline — TRAINING DATA ONLY

This is the single most important leakage rule in this phase: the median
(for imputation) and mean/std (for scaling) are computed exclusively from
the training split.


In [6]:
preprocessor, numeric_cols, categorical_cols = pp.fit_preprocessing_pipeline(patient_df, train_ids)
print("Pipeline fit complete.")
print(f"Numeric columns: {len(numeric_cols)} | Categorical columns: {len(categorical_cols)}")


2026-09-14 06:22:59,047 | INFO | src.preprocessing | Fitting preprocessing pipeline on 27560 training patients (458 numeric + 3 categorical features).
INFO:src.preprocessing:Fitting preprocessing pipeline on 27560 training patients (458 numeric + 3 categorical features).


Pipeline fit complete.
Numeric columns: 458 | Categorical columns: 3


## 4. Transform all three splits using the SAME fitted pipeline

In [7]:
X_train, y_train, feature_names = pp.transform_split(preprocessor, patient_df, train_ids, numeric_cols, categorical_cols)
X_val, y_val, _ = pp.transform_split(preprocessor, patient_df, val_ids, numeric_cols, categorical_cols)
X_test, y_test, _ = pp.transform_split(preprocessor, patient_df, test_ids, numeric_cols, categorical_cols)

print("X_train:", X_train.shape, "| y_train positive rate:", y_train.mean().round(4))
print("X_val:  ", X_val.shape, "| y_val positive rate:  ", y_val.mean().round(4))
print("X_test: ", X_test.shape, "| y_test positive rate: ", y_test.mean().round(4))

print()
print("Any NaN remaining in X_train?", np.isnan(X_train).any())
print("Any NaN remaining in X_val?", np.isnan(X_val).any())
print("Any NaN remaining in X_test?", np.isnan(X_test).any())


X_train: (27560, 464) | y_train positive rate: 0.0626
X_val:   (5905, 464) | y_val positive rate:   0.0625
X_test:  (5906, 464) | y_test positive rate:  0.0626

Any NaN remaining in X_train? False
Any NaN remaining in X_val? False
Any NaN remaining in X_test? False


## 5. Save the fitted pipeline and transformed arrays

Saving the transformed arrays too (not just the pipeline) so Phase 13-16
model-training notebooks can load them directly without re-running
preprocessing each time.


In [8]:
pp.save_pipeline(preprocessor, numeric_cols, categorical_cols)

np.savez(
    config.PROCESSED_DIR / "model_ready_arrays.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
)
pd.Series(feature_names).to_csv(config.PROCESSED_DIR / "feature_names.csv", index=False, header=["feature_name"])

print("Saved:")
print(f" - {config.MODELS_DIR / 'preprocessing_pipeline.pkl'}")
print(f" - {config.PROCESSED_DIR / 'model_ready_arrays.npz'}")
print(f" - {config.PROCESSED_DIR / 'feature_names.csv'} ({len(feature_names)} feature names)")


2026-09-14 06:23:04,511 | INFO | src.preprocessing | Saved preprocessing pipeline to /content/drive/MyDrive/Early-Sepsis-Detection/models/preprocessing_pipeline.pkl
INFO:src.preprocessing:Saved preprocessing pipeline to /content/drive/MyDrive/Early-Sepsis-Detection/models/preprocessing_pipeline.pkl


Saved:
 - /content/drive/MyDrive/Early-Sepsis-Detection/models/preprocessing_pipeline.pkl
 - /content/drive/MyDrive/Early-Sepsis-Detection/data/processed/model_ready_arrays.npz
 - /content/drive/MyDrive/Early-Sepsis-Detection/data/processed/feature_names.csv (464 feature names)


---
### What to send back to Claude after running this notebook

- Section 1's feature/numeric/categorical column counts
- Section 2's correlation demonstration (this shows numerically why those
  columns had to be excluded)
- Section 4's shapes and confirmation that no NaN remains in any split

With that, **Phase 11 is complete**, and we move to **Phase 12 (Class
Imbalance Analysis)** — examining class weights and scale_pos_weight
options (NOT SMOTE) — before training the first models in **Phase 13
(Baseline Logistic Regression)**.
